# LightFM Baseline trong bài toán gợi ý phim với phản hồi ẩn

## Tóm tắt (Abstract)
Notebook này xây dựng baseline LightFM sử dụng hàm mất mát Weighted Approximate-Rank Pairwise (WARP) trên dữ liệu phản hồi ẩn. Mô hình kết hợp tương tác người dùng-item với đặc trưng nội dung cơ bản của item và đặc trưng hồ sơ tĩnh của người dùng, sau đó đánh giá bằng các chỉ số Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$.

## 1. Giới thiệu (Introduction)
LightFM được sử dụng làm baseline học máy chính vì có khả năng kết hợp lọc cộng tác với đặc trưng phụ (side information). Trong notebook này, các đặc trưng được giới hạn ở metadata cơ bản nhằm tạo mốc so sánh rõ ràng cho các biến thể SeRel-LightFM sử dụng embedding văn bản và embedding đồ thị tri thức.

| Thành phần | Mô tả |
|---|---|
| Mô hình | LightFM với WARP loss |
| Item features | Thể loại, chủ đề, quốc gia, năm phát hành và thời lượng |
| User features | Hồ sơ tĩnh và các biến phân nhóm hành vi |
| Đánh giá | LOO, mask item đã xuất hiện trong train |


## 2. Phương pháp nghiên cứu (Methodology)

### 2.1 Thiết lập môi trường và cấu hình
Cell cấu hình khai báo nguồn dữ liệu, checkpoint, ngưỡng phản hồi ẩn $\tau$, siêu tham số LightFM và danh sách cutoff Top-$K$. Các lựa chọn này được giữ cố định để làm mốc so sánh với các biến thể ablation và mô hình đề xuất.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp thư viện và khai báo toàn bộ cấu hình thực nghiệm của notebook.
# - Đầu vào chính: URL dữ liệu, thư mục checkpoint, ngưỡng phản hồi ẩn và siêu tham số mô hình.
# - Kết quả tạo ra: các hằng số cấu hình được các cell phía sau sử dụng thống nhất.
# - Lưu ý: cấu hình thuộc biến thể LightFM Baseline, do đó cần giữ cố định khi so sánh với notebook khác.

import os, pickle, ast, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Nguồn dữ liệu ───────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

CHECKPOINT_DIR = "checkpoints_baseline"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Cấu hình thực nghiệm ──────────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7       # Binary implicit: rate >= này → positive
MIN_RATINGS        = 10      # Lọc user có ít hơn N ratings (cần >= 3 để LOO hợp lệ)

# LightFM
NO_COMPONENTS  = 64
LEARNING_RATE  = 0.05
ITEM_ALPHA     = 1e-6
USER_ALPHA     = 1e-6
MAX_EPOCHS     = 30
PATIENCE       = 5
NUM_THREADS    = 4
K_VALUES       = (5, 10, 20, 50)

print(f"[BASELINE] Cấu hình: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}, "
      f"lr={LEARNING_RATE}, patience={PATIENCE}")
print(f"[BASELINE] Split : Leave-One-Out (LOO) per user")
print(f"[BASELINE] Features: Tier 1 (genre/topic/country/year/duration) + Tier 3 (user metadata)")
print(f"[BASELINE] NO Tier 2 (TinyBERT / KGE / PCA)")

[BASELINE] Config: threshold=7, components=64, lr=0.05, patience=5
[BASELINE] Split : Leave-One-Out (LOO) per user
[BASELINE] Features: Tier 1 (genre/topic/country/year/duration) + Tier 3 (user metadata)
[BASELINE] NO Tier 2 (TinyBERT / KGE / PCA)


### 2.2 Các hàm hỗ trợ
Các hàm hỗ trợ thực hiện chuẩn hóa văn bản, token hóa các trường phân tách bằng dấu `|`, lọc token hiếm và xử lý giá trị thiếu. Những bước này tạo đầu vào ổn định cho quá trình xây dựng đặc trưng.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa các hàm tiền xử lý văn bản và token dùng lại trong notebook.
# - Đầu vào chính: các cột metadata dạng chuỗi, thường chứa token phân tách bằng dấu `|`.
# - Kết quả tạo ra: chuỗi đã làm sạch, danh sách token và tập token đủ phổ biến để làm đặc trưng.
# - Lưu ý: các hàm này chỉ chuẩn hóa dữ liệu; không học tham số từ validation hoặc test.

def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col


def tokenize(series, sep='|'):
    """Split pipe-separated Series → list-of-lists. NaN → []."""
    return [
        [t.strip() for t in s.split(sep) if t.strip()]
        if isinstance(s, str) else []
        for s in series
    ]


def filter_rare(token_lists, min_count):
    """Loại token xuất hiện < min_count lần."""
    if min_count <= 1:
        return token_lists
    counter = Counter()
    for toks in token_lists:
        counter.update(set(toks))
    keep = {t for t, c in counter.items() if c >= min_count}
    return [[t for t in toks if t in keep] for toks in token_lists]


def fix_plot(p):
    """Fix byte-string encoded plots và encoding artifacts."""
    if not isinstance(p, str) or p.strip() in ('', 'nan'):
        return ""
    if p.startswith("b'") or p.startswith('b"'):
        try:
            decoded = ast.literal_eval(p)
            if isinstance(decoded, bytes):
                return decoded.decode('utf-8', errors='replace')
        except Exception:
            try:
                decoded = ast.literal_eval(p)
                if isinstance(decoded, bytes):
                    return decoded.decode('latin-1', errors='replace')
            except Exception:
                return p[2:-1]
    return p


print("Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot")

Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot


### 2.3 Nạp dữ liệu
Các bảng `movies.csv`, `ratings.csv` và `user_profiles.csv` được nạp vào bộ nhớ. Cột thời gian được chuẩn hóa để phục vụ chia dữ liệu Leave-One-Out theo thứ tự tương tác.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp ba bảng dữ liệu gốc gồm rating, hồ sơ người dùng và metadata phim.
# - Đầu vào chính: các đường dẫn `RATINGS_URL`, `USERS_URL` và `MOVIES_URL`.
# - Kết quả tạo ra: `ratings_df`, `users_df`, `movies_df` và cột thời gian đã được chuẩn hóa.
# - Lưu ý: kiểm tra kích thước bảng sau khi nạp giúp phát hiện sớm lỗi đọc dữ liệu.

ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


### 2.4 Lọc dữ liệu hợp lệ
Các rating không có hồ sơ người dùng, người dùng có ít hơn `MIN_RATINGS` rating và movie không tồn tại trong metadata được loại bỏ. Bước này giảm nhiễu và bảo đảm LOO split khả thi.


In [ ]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

# 2.1 Lọc orphan users (có rating nhưng KHÔNG có profile)
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS ratings
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại trong movies.csv
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


### 2.5 Chuẩn hóa văn bản metadata
Các trường văn bản được làm sạch để khắc phục lỗi encoding và định dạng byte-string. Vì metadata là thuộc tính tĩnh của item, bước này không sử dụng thông tin từ nhãn validation/test.


In [ ]:
# [Giải thích cell]
# - Mục đích: làm sạch lỗi encoding trong metadata phim trước khi token hóa hoặc hiển thị.
# - Đầu vào chính: các cột văn bản như `genres`, `topics`, `actors`, `directors` và `plot`.
# - Kết quả tạo ra: metadata nhất quán hơn, giảm nhiễu cho đặc trưng nội dung và text encoder.
# - Lưu ý: đây là xử lý thuộc tính item, không dùng nhãn validation/test để học mô hình.

# 3.1 Fix encoding artifacts cho pipe-separated columns
for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

# 3.2 Fix byte-string plots (không dùng cho embedding, chỉ cleanup)
movies_df["plot_clean"] = movies_df["plot"].apply(fix_plot)

print("Text cleanup hoàn tất.")
print(f"Plot NaN/empty: {(movies_df['plot_clean'] == '').sum():,} / {len(movies_df):,}")

Text cleanup hoàn tất.
Plot NaN/empty: 2,442 / 78,628


### 2.6 Chia dữ liệu Leave-One-Out và áp dụng threshold
Quy trình dữ liệu được tách thành hai giai đoạn rõ ràng:

1. **Chia Leave-One-Out trên rating gốc:** `ratings_df` sau khi lọc hợp lệ vẫn giữ đầy đủ giá trị rating. Với mỗi người dùng, rating được sắp xếp theo thời gian; rating mới nhất được đưa vào `test_df`, rating liền trước được đưa vào `valid_df`, và toàn bộ rating còn lại được đưa vào `train_df`.
2. **Áp dụng threshold sau khi đã chia:** các split `train_df`, `valid_df` và `test_df` vẫn là dữ liệu rating gốc. Khi xây dựng tương tác phản hồi ẩn, chỉ các dòng thỏa $r_{u,i} \geq \tau$ mới được chuyển thành tương tác dương.

Điều này có nghĩa là threshold không được dùng để quyết định dòng nào thuộc train, validation hay test. Threshold chỉ quyết định dòng nào trở thành tương tác dương khi huấn luyện hoặc đánh giá. Cách làm này giữ đúng thứ tự thời gian của hành vi người dùng và tránh làm thay đổi tương tác mới nhất trước khi chia dữ liệu.


In [ ]:
# [Giải thích cell]
# - Mục đích: chia dữ liệu Leave-One-Out trên rating gốc, trước khi áp dụng threshold phản hồi ẩn.
# - Cách làm: sắp xếp rating theo thời gian cho từng user; lấy rating mới nhất làm test, rating liền trước làm validation, phần còn lại làm train.
# - Kết quả tạo ra: `train_df`, `valid_df`, `test_df`; các bảng này vẫn giữ rating gốc, gồm cả rating cao và thấp.
# - Lưu ý: threshold `rate >= POSITIVE_THRESHOLD` chỉ được áp dụng ở các bước xây dựng interaction/evaluation sau đó.

ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx = []
valid_idx = []
test_idx  = []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")
print(f"\nDate range train: {train_df['date'].min().date()} – {train_df['date'].max().date()}")
print(f"Date range valid: {valid_df['date'].min().date()} – {valid_df['date'].max().date()}")
print(f"Date range test : {test_df['date'].min().date()} – {test_df['date'].max().date()}")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)

Date range train: 2002-08-14 – 2021-03-31
Date range valid: 2011-12-06 – 2021-04-01
Date range test : 2011-12-08 – 2021-04-01


### 2.7 Phân loại item warm/cold
Item được phân loại là warm nếu xuất hiện trong train và cold nếu chỉ xuất hiện ở validation/test. Phân tích này giúp đánh giá khả năng tận dụng đặc trưng phụ của LightFM trong bối cảnh item cold-start.


In [ ]:
# [Giải thích cell]
# - Mục đích: tách item trong tập kiểm thử thành warm và cold theo sự xuất hiện trong train.
# - Đầu vào chính: `train_df` và `test_df`.
# - Kết quả tạo ra: `test_warm_df`, `test_cold_df` cùng các phiên bản positive nếu có.
# - Lưu ý: phân tách này giúp đánh giá riêng khả năng xử lý item cold-start.

movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"Unique items in valid : {valid_df['movie_id'].nunique():,}")
print(f"Unique items in test  : {test_df['movie_id'].nunique():,}")
print()
print(f"TEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)  "
      f"— {test_warm_df['movie_id'].nunique():,} unique movies")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)  "
      f"— {test_cold_df['movie_id'].nunique():,} unique movies")

Unique items in train : 75,115
Unique items in valid : 398
Unique items in test  : 407

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)  — 379 unique movies
TEST cold  :     28 ratings  (5.9%)  — 28 unique movies


### 2.8 Xây dựng item features cơ bản
Đặc trưng item gồm các biến phân loại `genre`, `topic`, `country` và các biến liên tục đã chuẩn hóa như `year`, `duration`. Đây là tập đặc trưng nội dung cơ sở, chưa sử dụng embedding văn bản hoặc embedding đồ thị tri thức.


In [ ]:
# [Giải thích cell]
# - Mục đích: xây dựng đặc trưng item có cấu trúc từ metadata phim.
# - Đầu vào chính: thể loại, chủ đề, quốc gia, năm phát hành và thời lượng.
# - Kết quả tạo ra: `item_feat_dicts`, trong đó mỗi movie_id ánh xạ tới các feature có trọng số.
# - Lưu ý: biến liên tục được chuẩn hóa để tránh lấn át đặc trưng phân loại.

# Token hóa các cột phân loại
genre_toks   = tokenize(movies_df["genres"])
topic_toks   = tokenize(movies_df["topics"])
country_toks = filter_rare(tokenize(movies_df["country_name"]), min_count=2)

# Tham số chuẩn hóa biến liên tục
year_min   = movies_df["year_published"].min()
year_range = movies_df["year_published"].max() - year_min
dur_cap    = movies_df["duration"].quantile(0.99)

# Xây dựng từ điển đặc trưng item: {movie_id: {feature_name: weight}}
item_feat_dicts = {}
for i in range(len(movies_df)):
    row = movies_df.iloc[i]
    mid = int(row["id"])
    feats = {}

    # ── Continuous features (normalized weight) ──
    if pd.notna(row["year_published"]) and year_range > 0:
        feats["year"] = (row["year_published"] - year_min) / year_range
    if pd.notna(row["duration"]) and dur_cap > 0:
        feats["duration"] = min(row["duration"] / dur_cap, 1.0)

    # ── Categorical features (weight = 1.0) ──
    for g in genre_toks[i]:
        feats[f"genre:{g}"] = 1.0
    for t in topic_toks[i]:
        feats[f"topic:{t}"] = 1.0
    for c in country_toks[i]:
        feats[f"country:{c}"] = 1.0

    item_feat_dicts[mid] = feats

# Tóm tắt kiểm tra
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

n_genre   = len({t for t in all_item_tags if t.startswith("genre:")})
n_topic   = len({t for t in all_item_tags if t.startswith("topic:")})
n_country = len({t for t in all_item_tags if t.startswith("country:")})
n_cont    = len({t for t in all_item_tags if t in ("year", "duration")})

print(f"[BASELINE] Đặc trưng item (Tier 1 only): {len(all_item_tags)} unique tags")
print(f"  genre     : {n_genre}")
print(f"  topic     : {n_topic}")
print(f"  country   : {n_country}")
print(f"  continuous: {n_cont} (year, duration)")
print(f"\n[BASELINE] Không có Tier 2 (TinyBERT / KGE / PCA) — tổng item features = {len(all_item_tags)}")

[BASELINE] Item features (Tier 1 only): 590 unique tags
  genre     : 19
  topic     : 395
  country   : 174
  continuous: 2 (year, duration)

[BASELINE] Không có Tier 2 (TinyBERT / KGE / PCA) — tổng item features = 590


### 2.9 Xây dựng user features từ hồ sơ tĩnh
Đặc trưng người dùng được tạo từ hồ sơ tĩnh, bao gồm các cờ nhị phân và biến đã phân nhóm. Các đặc trưng này không phụ thuộc vào lịch sử rating, nhờ đó tránh đưa thông tin tương tác validation/test vào mô hình.


In [ ]:
# [Giải thích cell]
# - Mục đích: chuyển hồ sơ người dùng thành đặc trưng rời rạc cho LightFM.
# - Đầu vào chính: các cột hành vi và hồ sơ tĩnh trong `users_df`.
# - Kết quả tạo ra: `user_feat_dicts`, ánh xạ user_id tới tập feature mô tả người dùng.
# - Lưu ý: đặc trưng user không lấy từ validation/test, giúp giảm nguy cơ rò rỉ dữ liệu.

def bin_preferred_hour(h):
    if pd.isna(h):       return "hour:unknown"
    if 5 <= h <= 11:     return "hour:morning"
    elif 12 <= h <= 17:  return "hour:afternoon"
    elif 18 <= h <= 22:  return "hour:evening"
    else:                return "hour:night"

def bin_account_year(y):
    if pd.isna(y):     return "acc:unknown"
    if y < 2010:       return "acc:pre2010"
    elif y <= 2013:    return "acc:2010_2013"
    else:              return "acc:post2013"

users_df["hour_bin"]     = users_df["preferred_hour"].apply(bin_preferred_hour)
users_df["acc_year_bin"] = users_df["account_creation_year"].apply(bin_account_year)
users_df["weekday_bin"]  = "weekday:" + users_df["preferred_weekday"].astype(str)

print("Hour bins    :", users_df["hour_bin"].value_counts().to_dict())
print("Acc year bins:", users_df["acc_year_bin"].value_counts().to_dict())
print("Weekday bins :", users_df["weekday_bin"].value_counts().to_dict())

BINARY_FLAGS = ["night_owl", "early_bird", "weekend_tweeter", "week_tweeter", "geo_enabled"]

user_feat_dicts = {}
for _, row in users_df.iterrows():
    uid   = int(row["id"])
    feats = {}

    for col in BINARY_FLAGS:
        if row.get(col, 0) == 1:
            feats[f"user:{col}"] = 1.0

    feats[row["hour_bin"]]     = 1.0
    feats[row["acc_year_bin"]] = 1.0
    feats[row["weekday_bin"]]  = 1.0

    user_feat_dicts[uid] = feats

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_binary  = len({t for t in all_user_tags if t.startswith("user:")})
n_hour    = len({t for t in all_user_tags if t.startswith("hour:")})
n_acc     = len({t for t in all_user_tags if t.startswith("acc:")})
n_weekday = len({t for t in all_user_tags if t.startswith("weekday:")})

print(f"\n[BASELINE] Tier 3 user features: {len(all_user_tags)} unique tags")
print(f"  binary flags : {n_binary}")
print(f"  hour bins    : {n_hour}")
print(f"  acc year bins: {n_acc}")
print(f"  weekday bins : {n_weekday}")
print(f"  Users với features: {len(user_feat_dicts)} / {users_df['id'].nunique()}")
print(f"\nSample user (id={list(user_feat_dicts.keys())[0]}): {list(user_feat_dicts.values())[0]}")

Hour bins    : {'hour:evening': 179, 'hour:afternoon': 127, 'hour:night': 101, 'hour:morning': 67}
Acc year bins: {'acc:2010_2013': 349, 'acc:pre2010': 73, 'acc:post2013': 52}
Weekday bins : {'weekday:0': 119, 'weekday:6': 90, 'weekday:2': 80, 'weekday:3': 64, 'weekday:1': 57, 'weekday:4': 40, 'weekday:5': 24}

[BASELINE] Tier 3 user features: 19 unique tags
  binary flags : 5
  hour bins    : 4
  acc year bins: 3
  weekday bins : 7
  Users với features: 474 / 474

Sample user (id=103007): {'user:night_owl': 1.0, 'user:geo_enabled': 1.0, 'hour:evening': 1.0, 'acc:pre2010': 1.0, 'weekday:1': 1.0}


### 2.10 Lưu checkpoint tiền xử lý
Các `DataFrame`, từ điển đặc trưng và cấu hình được lưu lại để có thể tái lập giai đoạn huấn luyện mà không cần chạy lại toàn bộ tiền xử lý.


In [ ]:
# [Giải thích cell]
# - Mục đích: lưu các artifact sau tiền xử lý để tái sử dụng ở các lần chạy sau.
# - Đầu vào chính: dataframes, từ điển đặc trưng, cấu hình và artifact embedding nếu có.
# - Kết quả tạo ra: một file `.pkl` trong thư mục checkpoint.
# - Lưu ý: checkpoint giúp tránh chạy lại các bước tốn thời gian như encoder, KMeans hoặc TransE.

BASELINE_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "baseline_preprocessed.pkl")

baseline_artifacts = {
    # Các bảng dữ liệu
    "ratings_df":    ratings_df,
    "users_df":      users_df,
    "movies_df":     movies_df,
    "train_df":      train_df,
    "valid_df":      valid_df,
    "test_df":       test_df,
    "test_warm_df":  test_warm_df,
    "test_cold_df":  test_cold_df,
    # Từ điển đặc trưng
    "item_feat_dicts": item_feat_dicts,
    "user_feat_dicts": user_feat_dicts,
    "all_item_tags":   all_item_tags,
    "all_user_tags":   all_user_tags,
    # Cấu hình
    "config": {
        "POSITIVE_THRESHOLD": POSITIVE_THRESHOLD,
        "MIN_RATINGS":        MIN_RATINGS,
        "NO_COMPONENTS":      NO_COMPONENTS,
        "LEARNING_RATE":      LEARNING_RATE,
        "ITEM_ALPHA":         ITEM_ALPHA,
        "USER_ALPHA":         USER_ALPHA,
        "MAX_EPOCHS":         MAX_EPOCHS,
        "PATIENCE":           PATIENCE,
        "NUM_THREADS":        NUM_THREADS,
        "K_VALUES":           K_VALUES,
    },
}

with open(BASELINE_CHECKPOINT, "wb") as f:
    pickle.dump(baseline_artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(BASELINE_CHECKPOINT) / (1024 * 1024)
print(f"Saved baseline checkpoint: {BASELINE_CHECKPOINT}  ({size_mb:.1f} MB)")
print(f"Keys: {list(baseline_artifacts.keys())}")

Saved baseline checkpoint: checkpoints_baseline/baseline_preprocessed.pkl  (146.9 MB)
Keys: ['ratings_df', 'users_df', 'movies_df', 'train_df', 'valid_df', 'test_df', 'test_warm_df', 'test_cold_df', 'item_feat_dicts', 'user_feat_dicts', 'all_item_tags', 'all_user_tags', 'config']


### 2.11 Nạp checkpoint tiền xử lý
Khi `LOAD_BASELINE = True`, notebook nạp lại checkpoint đã lưu và tiếp tục từ bước khởi tạo `Dataset` của LightFM. Bước này hỗ trợ tái lập thực nghiệm nhanh hơn.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp lại checkpoint đã lưu để tiếp tục từ bước xây dựng ma trận LightFM.
# - Đầu vào chính: file `.pkl` chứa dữ liệu và đặc trưng đã tiền xử lý.
# - Kết quả tạo ra: các biến notebook được khôi phục về trạng thái sau preprocessing.
# - Lưu ý: cần bảo đảm checkpoint khớp với đúng biến thể thực nghiệm đang chạy.

import os, pickle, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

LOAD_BASELINE      = False   # Đặt True nếu đã có checkpoint
CHECKPOINT_DIR     = "checkpoints_baseline"
BASELINE_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "baseline_preprocessed.pkl")

if LOAD_BASELINE and os.path.exists(BASELINE_CHECKPOINT):
    with open(BASELINE_CHECKPOINT, "rb") as f:
        art = pickle.load(f)

    ratings_df      = art["ratings_df"]
    users_df        = art["users_df"]
    movies_df       = art["movies_df"]
    train_df        = art["train_df"]
    valid_df        = art["valid_df"]
    test_df         = art["test_df"]
    test_warm_df    = art["test_warm_df"]
    test_cold_df    = art["test_cold_df"]
    item_feat_dicts = art["item_feat_dicts"]
    user_feat_dicts = art["user_feat_dicts"]
    all_item_tags   = art["all_item_tags"]
    all_user_tags   = art["all_user_tags"]

    cfg = art["config"]
    POSITIVE_THRESHOLD = cfg["POSITIVE_THRESHOLD"]
    MIN_RATINGS        = cfg["MIN_RATINGS"]
    NO_COMPONENTS      = cfg["NO_COMPONENTS"]
    LEARNING_RATE      = cfg["LEARNING_RATE"]
    ITEM_ALPHA         = cfg["ITEM_ALPHA"]
    USER_ALPHA         = cfg["USER_ALPHA"]
    MAX_EPOCHS         = cfg["MAX_EPOCHS"]
    PATIENCE           = cfg["PATIENCE"]
    NUM_THREADS        = cfg["NUM_THREADS"]
    K_VALUES           = cfg["K_VALUES"]

    print(f"Loaded baseline checkpoint from: {BASELINE_CHECKPOINT}")
    print(f"  Train     : {len(train_df):>7,} | {train_df['id'].nunique()} users")
    print(f"  Valid     : {len(valid_df):>7,} | {valid_df['id'].nunique()} users")
    print(f"  Test      : {len(test_df):>7,} | {test_df['id'].nunique()} users")
    print(f"  Test warm : {len(test_warm_df):>7,} | {test_warm_df['id'].nunique()} users")
    print(f"  Test cold : {len(test_cold_df):>7,} | {test_cold_df['id'].nunique()} users")
    print(f"  Đặc trưng item : {len(all_item_tags)} (Tier 1 only)")
    print(f"  Đặc trưng user : {len(all_user_tags)} (Tier 3)")
else:
    if LOAD_BASELINE:
        print(f"⚠ Checkpoint không tồn tại: {BASELINE_CHECKPOINT}")
        print("  Hãy chạy Bước 0→8 trước.")
    else:
        print("LOAD_BASELINE=False — dùng kết quả từ preprocessing ở trên.")

LOAD_BASELINE=False — dùng kết quả từ preprocessing ở trên.


### 2.12 Khởi tạo `Dataset` của LightFM
Đối tượng `Dataset` của LightFM được fit trên toàn bộ catalogue hợp lệ:

- `users=users_df["id"]`: toàn bộ user còn lại sau bước lọc hợp lệ.
- `items=movies_df["id"]`: toàn bộ movie metadata sau bước lọc hợp lệ.
- `user_features` và `item_features`: toàn bộ tên feature đã tạo từ metadata, text clusters và/hoặc KGE clusters.

Bước này chỉ khai báo không gian user, item và feature cho LightFM. Mô hình chưa học từ validation/test tại đây. Việc huấn luyện mô hình chỉ bắt đầu khi gọi `model.fit_partial(...)` với `train_interactions`, vốn được tạo từ các tương tác dương trong `train_df`.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài LightFM, thư viện được dùng để huấn luyện mô hình gợi ý lai.
# - Lưu ý: cell này không huấn luyện mô hình; chỉ bảo đảm package có sẵn trong runtime.

pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831127 sha256=1ad55f97eea4d1f100ee51df59c5e54f0ec9a6c8de8324f791cdfb7ee8f3fe40
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [ ]:
# [Giải thích cell]
# - Mục đích: fit không gian user/item/feature cho LightFM và tạo sparse feature matrices.
# - Đầu vào catalogue: toàn bộ `users_df` và `movies_df` sau lọc hợp lệ.
# - Đầu vào feature: `user_feat_dicts` và `item_feat_dicts`, có thể gồm metadata, text clusters và KGE clusters.
# - Lưu ý: cell này chưa huấn luyện mô hình; huấn luyện chỉ dùng `train_interactions` ở bước sau.

from lightfm import LightFM
from lightfm.data import Dataset

# ── Thu thập toàn bộ tên đặc trưng ──
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

# ── Fit đối tượng Dataset ──
dataset = Dataset()
dataset.fit(
    users=users_df["id"].tolist(),
    items=movies_df["id"].tolist(),
    user_features=list(all_user_tags),
    item_features=list(all_item_tags),
)

# ── Xây dựng ma trận đặc trưng ──
item_features_raw = [(mid, feats) for mid, feats in item_feat_dicts.items()]
user_features_raw = [(uid, feats) for uid, feats in user_feat_dicts.items()]

item_feature_matrix = dataset.build_item_features(item_features_raw, normalize=False)
user_feature_matrix = dataset.build_user_features(user_features_raw, normalize=False)

print(f"User feature matrix : {user_feature_matrix.shape}")
print(f"Item feature matrix : {item_feature_matrix.shape}")
print(f"Item feature density: {item_feature_matrix.nnz / np.prod(item_feature_matrix.shape) * 100:.4f}%")
print(f"\n[BASELINE] Total user features: {len(all_user_tags)} (Tier 3 only)")
print(f"[BASELINE] Total item features: {len(all_item_tags)} (Tier 1 only — no Tier 2)")

User feature matrix : (474, 493)
Item feature matrix : (78628, 79218)
Item feature density: 0.0096%

[BASELINE] Total user features: 19 (Tier 3 only)
[BASELINE] Total item features: 590 (Tier 1 only — no Tier 2)


### 2.13 Xây dựng interactions nhị phân
Các bảng `train_df`, `valid_df` và `test_df` vẫn chứa rating gốc sau khi chia LOO. Tại bước này, threshold mới được áp dụng để tạo ma trận phản hồi ẩn:

$$
R_{u,i} = \mathbb{1}(r_{u,i} \geq \tau).
$$

`train_interactions` được tạo từ các rating dương trong `train_df` và là dữ liệu duy nhất dùng để huấn luyện LightFM. `valid_inter` và `test_inter` chỉ được dùng để early stopping/đánh giá, không được đưa vào `fit_partial`. Khi xếp hạng, các item đã xuất hiện trong raw `train_df` của user được mask để tránh gợi ý lại lịch sử đã biết.


In [ ]:
# [Giải thích cell]
# - Mục đích: áp dụng threshold sau LOO để tạo ma trận phản hồi ẩn cho từng split.
# - Dữ liệu train model: `train_interactions`, chỉ gồm các dòng trong `train_df` có `rate >= POSITIVE_THRESHOLD`.
# - Dữ liệu đánh giá: `valid_inter` và `test_inter`, cũng chỉ giữ held-out rating dương sau threshold.
# - Lưu ý: rating thấp hơn ngưỡng không là ground truth dương; raw `train_df` vẫn được dùng để mask item đã xem.

def build_interactions_binary(df, dataset, threshold=POSITIVE_THRESHOLD):
    """Binary implicit: rate >= ngưỡng → positive. Không trả sample_weight."""
    positives = df[df["rate"] >= threshold]
    if len(positives) == 0:
        return None
    pairs = [(int(r["id"]), int(r["movie_id"])) for _, r in positives.iterrows()]
    interactions, _ = dataset.build_interactions(pairs)
    return interactions

train_interactions = build_interactions_binary(train_df, dataset)
valid_inter        = build_interactions_binary(valid_df, dataset)
test_inter         = build_interactions_binary(test_df, dataset)
test_warm_inter    = build_interactions_binary(test_warm_df, dataset)
test_cold_inter    = build_interactions_binary(test_cold_df, dataset)

def nnz(m): return m.nnz if m is not None else 0

print(f"Threshold = {POSITIVE_THRESHOLD} (binary implicit, no sample_weight)")
print(f"Train interactions : {nnz(train_interactions):,}")
print(f"Valid interactions : {nnz(valid_inter):,}")
print(f"Test  interactions : {nnz(test_inter):,}")
print(f"  ├─ Warm items    : {nnz(test_warm_inter):,}")
print(f"  └─ Cold items    : {nnz(test_cold_inter):,}")

Threshold = 7 (binary implicit, no sample_weight)
Train interactions : 412,438
Valid interactions : 240
Test  interactions : 262
  ├─ Warm items    : 253
  └─ Cold items    : 9


## 3. Thực nghiệm và Kết quả (Experiments & Results)

### 3.1 Các chỉ số đánh giá Top-$K$
Các chỉ số Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$ được định nghĩa cho bài toán xếp hạng. Trong quá trình đánh giá, item đã xuất hiện trong train của người dùng được mask khỏi danh sách ứng viên.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa hàm đánh giá Top-K trên các held-out interactions dương.
# - Đầu vào chính: ma trận held-out sau threshold, `train_interactions` để mask lịch sử và mô hình sinh điểm.
# - Kết quả tạo ra: Precision@K, Recall@K, NDCG@K, HR@K và MRR@K.
# - Lưu ý: validation/test chỉ được dùng để đo hiệu năng; không cập nhật trọng số mô hình.

def evaluate_metrics(model, test_interactions, train_interactions,
                     user_features, item_features,
                     k_values=(5, 10, 20), num_threads=1):
    """
    Tính Precision@K, Recall@K, NDCG@K, HR@K, MRR@K cho nhiều K.
    Returns: dict {K: {metric: value}}
    """
    if test_interactions is None or test_interactions.nnz == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                     "hr": 0., "mrr": 0.} for k in k_values}

    test_csr  = test_interactions.tocsr()
    train_csr = train_interactions.tocsr()
    n_users, n_items = test_csr.shape
    max_k = max(k_values)

    accum = {k: {"precision": [], "recall": [], "ndcg": [],
                  "hr": [], "mrr": []} for k in k_values}

    for u in range(n_users):
        true_items = set(test_csr[u].indices.tolist())
        if not true_items:
            continue

        scores = model.predict(
            user_ids=u,
            item_ids=np.arange(n_items),
            user_features=user_features,
            item_features=item_features,
            num_threads=num_threads,
        )

        # Mask items đã có trong train
        train_items = train_csr[u].indices
        scores[train_items] = -np.inf

        top_idx = np.argpartition(-scores, max_k)[:max_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        relevance  = np.array([1.0 if i in true_items else 0.0 for i in top_idx])
        n_relevant = len(true_items)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    summary = {}
    for k in k_values:
        summary[k] = {m: float(np.mean(v)) if v else 0.0
                      for m, v in accum[k].items()}
    return summary


def print_metrics(metrics, label):
    print(f"\n=== {label} ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k, m in sorted(metrics.items()):
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")


print("Metrics defined: evaluate_metrics, print_metrics")

Metrics defined: evaluate_metrics, print_metrics


### 3.2 Huấn luyện LightFM với WARP
Mô hình được huấn luyện theo từng epoch bằng WARP loss và chọn checkpoint tốt nhất theo NDCG@10 trên tập validation. Bảng 1 là lịch sử huấn luyện được sinh từ cell bên dưới.


In [ ]:
# [Giải thích cell]
# - Mục đích: huấn luyện LightFM bằng WARP trên dữ liệu train đã nhị phân hóa.
# - Đầu vào huấn luyện duy nhất: `train_interactions` cùng ma trận feature user/item.
# - Validation: `valid_inter` chỉ dùng để tính NDCG@10 và early stopping sau mỗi epoch.
# - Lưu ý: `test_inter` không tham gia huấn luyện hay chọn epoch.

model = LightFM(
    loss="warp",
    no_components=NO_COMPONENTS,
    learning_rate=0.05,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=42,
)

best_ndcg10      = -1.0
patience_counter = 0
history          = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.fit_partial(
        interactions=train_interactions,
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
        epochs=1,
        num_threads=1,
        verbose=False,
    )

    valid_metrics = evaluate_metrics(
        model, valid_inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=2,
    )

    ndcg10 = valid_metrics[10]["ndcg"]
    history.append({
        "epoch": epoch,
        **{f"ndcg@{k}": valid_metrics[k]["ndcg"] for k in K_VALUES}
    })
    print(f"Epoch {epoch:>2} | "
          + " | ".join(f"NDCG@{k}: {valid_metrics[k]['ndcg']:.4f}" for k in K_VALUES))

    if ndcg10 > best_ndcg10:
        best_ndcg10      = ndcg10
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\n[BASELINE] Best Valid NDCG@10: {best_ndcg10:.4f}")

Epoch  1 | NDCG@5: 0.0094 | NDCG@10: 0.0106 | NDCG@20: 0.0127 | NDCG@50: 0.0229
Epoch  2 | NDCG@5: 0.0120 | NDCG@10: 0.0120 | NDCG@20: 0.0130 | NDCG@50: 0.0236
Epoch  3 | NDCG@5: 0.0136 | NDCG@10: 0.0163 | NDCG@20: 0.0173 | NDCG@50: 0.0246
Epoch  4 | NDCG@5: 0.0190 | NDCG@10: 0.0203 | NDCG@20: 0.0224 | NDCG@50: 0.0249
Epoch  5 | NDCG@5: 0.0151 | NDCG@10: 0.0180 | NDCG@20: 0.0233 | NDCG@50: 0.0280
Epoch  6 | NDCG@5: 0.0151 | NDCG@10: 0.0203 | NDCG@20: 0.0235 | NDCG@50: 0.0266
Epoch  7 | NDCG@5: 0.0225 | NDCG@10: 0.0239 | NDCG@20: 0.0259 | NDCG@50: 0.0314
Epoch  8 | NDCG@5: 0.0235 | NDCG@10: 0.0247 | NDCG@20: 0.0268 | NDCG@50: 0.0292
Epoch  9 | NDCG@5: 0.0185 | NDCG@10: 0.0239 | NDCG@20: 0.0239 | NDCG@50: 0.0289
Epoch 10 | NDCG@5: 0.0183 | NDCG@10: 0.0224 | NDCG@20: 0.0254 | NDCG@50: 0.0313
Epoch 11 | NDCG@5: 0.0121 | NDCG@10: 0.0192 | NDCG@20: 0.0214 | NDCG@50: 0.0256
Epoch 12 | NDCG@5: 0.0206 | NDCG@10: 0.0231 | NDCG@20: 0.0262 | NDCG@50: 0.0279
Epoch 13 | NDCG@5: 0.0203 | NDCG@10: 0.0

In [ ]:
# [Giải thích cell]
# - Mục đích: hiển thị lịch sử huấn luyện dưới dạng bảng để quan sát quá trình hội tụ.
# - Đầu vào chính: danh sách `history` được ghi lại sau mỗi epoch.
# - Kết quả tạo ra: `history_df`, bảng thể hiện loss/metric validation theo epoch.

history_df = pd.DataFrame(history)
display(history_df)

,epoch,ndcg@5,ndcg@10,ndcg@20,ndcg@50
0,1,0.009368,0.010572,0.012698,0.022868
1,2,0.012029,0.012029,0.013048,0.023636
2,3,0.013591,0.016330,0.017349,0.024606
3,4,0.019007,0.020261,0.022429,0.024937
4,5,0.015129,0.018002,0.023325,0.028047
5,6,0.015129,0.020291,0.023537,0.026628
6,7,0.022519,0.023908,0.025876,0.031369
7,8,0.023479,0.024684,0.026829,0.029219
8,9,0.018461,0.023853,0.023853,0.028901
9,10,0.018279,0.022356,0.025445,0.031313


### 3.3 Đánh giá cuối trên validation và test
Cell đánh giá báo cáo kết quả trên validation, test-full, test-warm và test-cold. Bảng 2 cho phép so sánh hiệu năng tổng thể và khả năng xử lý item cold-start của baseline LightFM.


In [ ]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

print("=" * 65)
print("FINAL EVALUATION — BASELINE (Tier 1 + Tier 3, NO Tier 2)")
print(f"Cấu hình: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}")
print(f"Đặc trưng item: {len(all_item_tags)} tags (genre/topic/country/year/duration)")
print(f"Đặc trưng user: {len(all_user_tags)} tags (binary flags + hour/acc_year/weekday)")
print("=" * 65)

for label, inter in [
    ("VALID",     valid_inter),
    ("TEST",      test_inter),
    ("TEST_WARM", test_warm_inter),
    ("TEST_COLD", test_cold_inter),
]:
    metrics = evaluate_metrics(
        model, inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )
    print_metrics(metrics, label)

FINAL EVALUATION — BASELINE (Tier 1 + Tier 3, NO Tier 2)
Config: threshold=7, components=64
Item features: 590 tags (genre/topic/country/year/duration)
User features: 19 tags (binary flags + hour/acc_year/weekday)

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0050 |   0.0250 |   0.0203 |   0.0250 |   0.0187
  10 |   0.0033 |   0.0333 |   0.0228 |   0.0333 |   0.0197
  20 |   0.0025 |   0.0500 |   0.0272 |   0.0500 |   0.0210
  50 |   0.0013 |   0.0667 |   0.0304 |   0.0667 |   0.0215

=== TEST ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0031 |   0.0153 |   0.0091 |   0.0153 |   0.0070
  10 |   0.0027 |   0.0267 |   0.0127 |   0.0267 |   0.0084
  20 |   0.0017 |   0.0344 |   0.0146 |   0.0344 |   0.0089
  50 |   0.0014 |   0.0687 |   0.0213 |   0.0687 |   0.0100

=== TEST_WARM ===
   K |     Prec |   

### 3.4 Suy luận cho một người dùng
Cell này minh họa quy trình sinh top-$N$ gợi ý cho một người dùng cụ thể. Các item đã có trong lịch sử train được loại bỏ để mô phỏng ngữ cảnh triển khai thực tế.


In [ ]:
# [Giải thích cell]
# - Mục đích: minh họa quy trình suy luận top-N cho một người dùng cụ thể.
# - Đầu vào chính: user_id, mô hình hoặc danh sách popularity, metadata phim và train history.
# - Kết quả tạo ra: bảng phim được đề xuất sau khi loại bỏ item người dùng đã thấy.
# - Lưu ý: đây là ví dụ định tính, không thay thế cho đánh giá định lượng Top-K.

def recommend_for_user(user_id, model, dataset, movies_df,
                       user_feature_matrix, item_feature_matrix,
                       train_df, n_recs=10):
    """Gợi ý top-N movies cho 1 user, loại trừ phim đã có trong train."""
    uid_map, _, iid_map, _ = dataset.mapping()
    if user_id not in uid_map:
        print(f"User {user_id} không tồn tại.")
        return pd.DataFrame()

    u_idx   = uid_map[user_id]
    n_items = item_feature_matrix.shape[0]

    scores = model.predict(
        user_ids=u_idx,
        item_ids=np.arange(n_items),
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
    )

    seen = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    seen_indices = [iid_map[m] for m in seen if m in iid_map]
    scores[seen_indices] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    idx2movie     = {v: k for k, v in iid_map.items()}
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["predicted_score"] = [scores[iid_map[mid]] for mid in result["id"]]
    return result.sort_values("predicted_score", ascending=False)


# Minh họa suy luận
sample_user = train_df["id"].iloc[0]
print(f"[BASELINE] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user(
    user_id=sample_user, model=model, dataset=dataset,
    movies_df=movies_df,
    user_feature_matrix=user_feature_matrix,
    item_feature_matrix=item_feature_matrix,
    train_df=train_df, n_recs=10,
)
print(recs.to_string(index=False))

[BASELINE] Gợi ý top-10 cho user_id = 103007

    id                                  original_title                               genres  year_published  rate  predicted_score
186830                                             300   Acción|Bélico|Aventuras|Fantástico          2006.0   7.2      -296.132690
716669                          Breakfast at Tiffany's                Romance|Drama|Comedia          1961.0   7.7      -296.381714
491709                                         Amadeus                                Drama          1984.0   7.7      -296.524689
595319 Master and Commander: The Far Side of the World                     Aventuras|Acción          2003.0   7.0      -296.538666
144674                                      Persepolis                      Animación|Drama          2007.0   7.8      -296.736267
523268             Kokaku Kidotai (Ghost in the Shell)     Animación|Ciencia ficción|Acción          1995.0   7.5      -296.785583
173696                               

## 4. Kết luận (Conclusion)
LightFM Baseline cung cấp mốc so sánh học máy có sử dụng đặc trưng phụ, nằm giữa các baseline đơn giản và SeRel-LightFM đầy đủ. Kết quả từ notebook này giúp xác định mức cải thiện đến từ việc bổ sung embedding văn bản và embedding đồ thị tri thức trong các notebook tiếp theo.
